In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.3 MB/s eta 0:00:00


In [4]:
# =========================
# Script 2: Evaluación cuantitativa completa
#
# BLOQUE A — mAP@50 y mAP@50:95 a nivel de frame (solo positivos)
# BLOQUE B — Clasificación a nivel de clip (positivos + negativos)
#             métricas: accuracy, precisión, recall, F1, matriz de confusión
# BLOQUE C — Análisis por categoría negativa (N1–N12)
# =========================
import json
import os
import shutil
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
from ultralytics import YOLO

# -------- CONFIG --------
POS_LIST   = "/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_test.txt"
NEG_LIST   = "/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/no_gun_test.txt"
OUT_DIR    = "/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/evaluation"

WEAPON_DET_WEIGHTS = "/content/drive/MyDrive/TFM/experiments/weapon_det/yolov8m_weapons_B_e50_640/weights/best.pt"

IMG_SIZE     = 640
CONF_WEAPON  = 0.25
IOU_NMS      = 0.7
FRAME_STRIDE = 1
MAX_SECONDS  = 15    # None para clip completo

# Umbral de clasificación: mínimo de frames con arma detectada para considerar clip como positivo
DETECTION_THRESHOLD = 5   # prueba también con 3, 5, 10
# ------------------------

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# -------- Cargar modelo --------
weapon_model = YOLO(WEAPON_DET_WEIGHTS)

# ============================================================
# HELPERS
# ============================================================

def iou(boxA, boxB):
    """IoU entre dos boxes [x1,y1,x2,y2]."""
    xA = max(boxA[0], boxB[0]);  yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]);  yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0:
        return 0.0
    aA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    aB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (aA + aB - inter)

def load_gt_boxes(label_json_path):
    """
    Devuelve dict {image_id: [(x1,y1,x2,y2), ...]} en formato xyxy.
    El JSON usa formato COCO: bbox = [x, y, w, h].
    """
    with open(label_json_path) as f:
        data = json.load(f)
    gt = defaultdict(list)
    for ann in data["annotations"]:
        x, y, w, h = ann["bbox"]
        gt[ann["image_id"]].append((x, y, x + w, y + h))
    return gt

def run_detector_on_video(video_path, fps_limit=None):
    """
    Ejecuta el detector frame a frame.
    Devuelve lista de (frame_idx, preds) donde preds = [(x1,y1,x2,y2,conf), ...]
    frame_idx es 1-based para coincidir con image_id del JSON.
    """
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    max_frames = n_frames
    if MAX_SECONDS is not None and fps and fps > 0:
        max_frames = min(max_frames, int(MAX_SECONDS * fps))

    results = []
    frame_i = 0
    while True:
        ok, frame = cap.read()
        if not ok or frame_i >= max_frames:
            break
        if frame_i % FRAME_STRIDE == 0:
            r = weapon_model.predict(frame, imgsz=IMG_SIZE, conf=CONF_WEAPON,
                                     iou=IOU_NMS, verbose=False, device='cuda')[0]
            preds = []
            if r.boxes is not None and len(r.boxes) > 0:
                for b in r.boxes:
                    x1, y1, x2, y2 = map(float, b.xyxy[0])
                    conf = float(b.conf[0])
                    preds.append((x1, y1, x2, y2, conf))
            results.append((frame_i + 1, preds))   # image_id es 1-based
        frame_i += 1

    cap.release()
    return results, max_frames

# ============================================================
# BLOQUE A — mAP a nivel de frame
# ============================================================
print("\n" + "="*60)
print("BLOQUE A — mAP a nivel de frame")
print("="*60)

IOU_THRESHOLDS = np.arange(0.5, 1.0, 0.05)   # 0.50, 0.55, ..., 0.95

pos_paths = [Path(l.strip()) for l in Path(POS_LIST).read_text().splitlines() if l.strip()]

all_tp    = defaultdict(list)   # iou_thr -> lista de 1/0
all_fp    = defaultdict(list)
total_gt  = 0

for vp in pos_paths:
    if not vp.exists():
        print(f"  ❌ No existe: {vp}")
        continue

    label_path = vp.parent / "label.json"
    if not label_path.exists():
        print(f"  ⚠️  Sin label.json: {vp.parent.name}")
        continue

    clip_id  = vp.parent.name
    local_in = f"/content/{clip_id}_eval.mp4"
    shutil.copy2(str(vp), local_in)

    gt_boxes = load_gt_boxes(label_path)
    total_gt += sum(len(v) for v in gt_boxes.values())

    frame_results, _ = run_detector_on_video(local_in)

    for (frame_idx, preds) in frame_results:
        gts = gt_boxes.get(frame_idx, [])

        for iou_thr in IOU_THRESHOLDS:
            matched_gt = set()
            for (px1, py1, px2, py2, conf) in sorted(preds, key=lambda x: -x[4]):
                best_iou, best_j = 0, -1
                for j, gt in enumerate(gts):
                    if j in matched_gt:
                        continue
                    s = iou((px1, py1, px2, py2), gt)
                    if s > best_iou:
                        best_iou, best_j = s, j
                if best_iou >= iou_thr and best_j >= 0:
                    all_tp[iou_thr].append(1)
                    all_fp[iou_thr].append(0)
                    matched_gt.add(best_j)
                else:
                    all_tp[iou_thr].append(0)
                    all_fp[iou_thr].append(1)

    os.remove(local_in)
    print(f"  ✅ {clip_id}")

# Calcular AP por umbral (precisión global simplificada)
aps = {}
for iou_thr in IOU_THRESHOLDS:
    tp = sum(all_tp[iou_thr])
    fp = sum(all_fp[iou_thr])
    fn = total_gt - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    aps[iou_thr] = {"precision": precision, "recall": recall}

map50    = aps[0.5]["precision"]
map5095  = np.mean([v["precision"] for v in aps.values()])

print(f"\n  Total GT boxes  : {total_gt}")
print(f"  mAP@50          : {map50:.4f}")
print(f"  mAP@50:95       : {map5095:.4f}")
print(f"  Precision@50    : {aps[0.5]['precision']:.4f}")
print(f"  Recall@50       : {aps[0.5]['recall']:.4f}")

# ============================================================
# BLOQUE B — Clasificación a nivel de clip
# ============================================================
print("\n" + "="*60)
print(f"BLOQUE B — Clasificación a nivel de clip (umbral={DETECTION_THRESHOLD} frames)")
print("="*60)

y_true, y_pred = [], []
clip_details   = []   # para bloque C

def classify_clip(video_path, label):
    local_in = f"/content/tmp_cls.mp4"
    shutil.copy2(str(video_path), local_in)
    frame_results, total = run_detector_on_video(local_in)
    os.remove(local_in)

    frames_with_gun = sum(1 for (_, preds) in frame_results if len(preds) > 0)
    predicted = 1 if frames_with_gun >= DETECTION_THRESHOLD else 0
    return predicted, frames_with_gun, total

# Positivos
print("\n  Procesando positivos...")
for vp in pos_paths:
    if not vp.exists():
        continue
    clip_id = vp.parent.name
    pred, n_det, n_total = classify_clip(vp, 1)
    y_true.append(1)
    y_pred.append(pred)
    category = clip_id.split("_")[0]   # PAH* → positivo, guardamos igual
    clip_details.append({"clip": clip_id, "true": 1, "pred": pred,
                          "det_frames": n_det, "total_frames": n_total,
                          "category": "POS"})
    print(f"    {clip_id}: {n_det}/{n_total} frames con arma → {'✅TP' if pred==1 else '❌FN'}")

# Negativos
neg_paths = [Path(l.strip()) for l in Path(NEG_LIST).read_text().splitlines() if l.strip()]
print(f"\n  Procesando negativos ({len(neg_paths)} clips)...")
for vp in neg_paths:
    if not vp.exists():
        continue
    clip_id  = vp.parent.name
    category = clip_id.split("_")[0]   # N1, N2, ...
    pred, n_det, n_total = classify_clip(vp, 0)
    y_true.append(0)
    y_pred.append(pred)
    clip_details.append({"clip": clip_id, "true": 0, "pred": pred,
                          "det_frames": n_det, "total_frames": n_total,
                          "category": category})
    print(f"    {clip_id}: {n_det}/{n_total} frames con arma → {'❌FP' if pred==1 else '✅TN'}")

# Métricas globales
y_true = np.array(y_true)
y_pred = np.array(y_pred)

TP = int(((y_true == 1) & (y_pred == 1)).sum())
TN = int(((y_true == 0) & (y_pred == 0)).sum())
FP = int(((y_true == 0) & (y_pred == 1)).sum())
FN = int(((y_true == 1) & (y_pred == 0)).sum())

accuracy  = (TP + TN) / len(y_true)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n  --- Métricas globales (clip-level) ---")
print(f"  Clips evaluados : {len(y_true)} ({TP+FN} pos / {TN+FP} neg)")
print(f"  Accuracy        : {accuracy:.4f}")
print(f"  Precision       : {precision:.4f}")
print(f"  Recall          : {recall:.4f}")
print(f"  F1              : {f1:.4f}")
print(f"\n  Matriz de confusión:")
print(f"                Pred POS   Pred NEG")
print(f"  Real POS    |   {TP:4d}   |   {FN:4d}  |")
print(f"  Real NEG    |   {FP:4d}   |   {TN:4d}  |")

# ============================================================
# BLOQUE C — Análisis por categoría negativa
# ============================================================
print("\n" + "="*60)
print("BLOQUE C — Falsos positivos por categoría negativa")
print("="*60)

neg_details = [d for d in clip_details if d["true"] == 0]
cat_stats   = defaultdict(lambda: {"total": 0, "fp": 0})

for d in neg_details:
    cat = d["category"]
    cat_stats[cat]["total"] += 1
    if d["pred"] == 1:
        cat_stats[cat]["fp"] += 1

cat_labels = {
    "N1": "Walking empty hands", "N2": "Jogging", "N3": "Running",
    "N4": "Sneaking empty hands", "N5": "Phone relaxed",
    "N6": "Phone looking", "N7": "Phone both hands",
    "N8": "Phone recording 1h", "N9": "Phone recording 2h",
    "N10": "Water bottle relaxed", "N11": "Drinking",
    "N12": "Holding heavy object"
}

print(f"\n  {'Cat':<5} {'Descripción':<28} {'FP':>4} {'Total':>6} {'FP%':>7}")
print("  " + "-"*55)
for cat in sorted(cat_stats, key=lambda x: int(x[1:])):
    s = cat_stats[cat]
    fp_rate = s["fp"] / s["total"] * 100 if s["total"] > 0 else 0
    desc = cat_labels.get(cat, "")
    print(f"  {cat:<5} {desc:<28} {s['fp']:>4} {s['total']:>6} {fp_rate:>6.1f}%")

# ============================================================
# GUARDAR RESULTADOS
# ============================================================
results_path = Path(OUT_DIR) / "evaluation_results_B.txt"
with open(results_path, "w") as f:
    f.write("=== EVALUACIÓN CUANTITATIVA ===\n\n")
    f.write(f"Modelo: {WEAPON_DET_WEIGHTS}\n")
    f.write(f"CONF_WEAPON={CONF_WEAPON} | IOU_NMS={IOU_NMS} | DETECTION_THRESHOLD={DETECTION_THRESHOLD}\n\n")

    f.write("--- BLOQUE A: mAP a nivel de frame ---\n")
    f.write(f"Total GT boxes : {total_gt}\n")
    f.write(f"mAP@50         : {map50:.4f}\n")
    f.write(f"mAP@50:95      : {map5095:.4f}\n")
    f.write(f"Precision@50   : {aps[0.5]['precision']:.4f}\n")
    f.write(f"Recall@50      : {aps[0.5]['recall']:.4f}\n\n")

    f.write("--- BLOQUE B: Clasificación clip-level ---\n")
    f.write(f"Accuracy   : {accuracy:.4f}\n")
    f.write(f"Precision  : {precision:.4f}\n")
    f.write(f"Recall     : {recall:.4f}\n")
    f.write(f"F1         : {f1:.4f}\n")
    f.write(f"TP={TP} TN={TN} FP={FP} FN={FN}\n\n")

    f.write("--- BLOQUE C: FP por categoría negativa ---\n")
    for cat in sorted(cat_stats, key=lambda x: int(x[1:])):
        s = cat_stats[cat]
        fp_rate = s["fp"] / s["total"] * 100 if s["total"] > 0 else 0
        f.write(f"  {cat}: {s['fp']}/{s['total']} ({fp_rate:.1f}%)\n")

print(f"\n✅ Resultados guardados en: {results_path}")
print("\n✅ Evaluación completa.")


BLOQUE A — mAP a nivel de frame
  ✅ PAH1_C1_P1_V1_HB_3
  ✅ PAH1_C1_P1_V1_HB_4
  ✅ PAH1_C1_P2_V1_HB_1
  ✅ PAH1_C1_P2_V1_HB_3
  ✅ PAH1_C1_P4_V1_HB_1
  ✅ PAH1_C1_P4_V1_HB_2
  ✅ PAH1_C2_P3_V1_HB_1
  ✅ PAH1_C2_P3_V1_HB_3
  ✅ PAH1_C2_P3_V2_HB_2
  ✅ PAH1_C2_P3_V2_HB_3
  ✅ PAH1_C2_P5_V1_HB_1
  ✅ PAH1_C2_P5_V1_HB_4
  ✅ PAH1_C2_P5_V2_HB_3
  ✅ PAH1_C2_P5_V2_HB_4
  ✅ PAH2_C1_P1_V1_HB_1
  ✅ PAH2_C1_P1_V1_HB_4
  ✅ PAH2_C1_P2_V1_HB_3
  ✅ PAH2_C1_P2_V1_HB_4
  ✅ PAH2_C1_P4_V1_HB_1
  ✅ PAH2_C1_P4_V1_HB_3
  ✅ PAH2_C2_P3_V1_HB_1
  ✅ PAH2_C2_P3_V1_HB_4
  ✅ PAH2_C2_P3_V2_HB_2
  ✅ PAH2_C2_P3_V2_HB_4
  ✅ PAH2_C2_P5_V1_HB_3
  ✅ PAH2_C2_P5_V1_HB_4
  ✅ PAH2_C2_P5_V2_HB_2
  ✅ PAH2_C2_P5_V2_HB_4
  ✅ PAH3_C1_P1_V1_HB_1
  ✅ PAH3_C1_P1_V1_HB_4
  ✅ PAH3_C1_P2_V1_HB_1
  ✅ PAH3_C1_P2_V1_HB_2
  ✅ PAH3_C1_P4_V1_HB_2
  ✅ PAH3_C1_P4_V1_HB_4
  ✅ PAH3_C2_P3_V1_HB_1
  ✅ PAH3_C2_P3_V1_HB_2
  ✅ PAH3_C2_P3_V2_HB_1
  ✅ PAH3_C2_P3_V2_HB_3
  ✅ PAH3_C2_P5_V1_HB_2
  ✅ PAH3_C2_P5_V1_HB_4
  ✅ PAH3_C2_P5_V2_HB_2
  ✅ PAH3_C2_P5_V2_HB_4
 